In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl-scientific-track")

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "starter_kit").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "Scientific Track" / "starter_kit").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR / "Scientific Track"))

from starter_kit import (
    animate_linecut,
    build_annni_hamiltonian,
    bkt_transition,
    exact_ground_state,
    ising_transition,
    kt_transition,
    noisy_entangling_layer,
    order_parameter_summary,
    plot_coupling_graph,
    plot_observable_heatmap,
    simple_noisy_energy,
)


In [ ]:
import pennylane as qml
import matplotlib.pyplot as plt
import numpy as np
from pennylane import numpy as pnp
import vqe


In [ ]:
import pennylane as qml
import numpy as np


def zz_gate(theta, wire_i, wire_j):
    qml.CNOT(wires=[wire_i, wire_j])
    qml.RZ(-2 * theta, wires=wire_j)
    qml.CNOT(wires=[wire_i, wire_j])


def annni_trotter_step(n_qubits, J2, h, dt, pbc=True):

    # --- Layer 1: Nearest-neighbor ZZ interactions ---
    # exp(+i * J1 * dt * Z_i Z_{i+1})  [note: H has -J1, so exponent flips sign]
    for i in range(n_qubits - 1):
        zz_gate(dt, i, i + 1)             # theta = +J1*dt  ←  -(-J1)*dt
    if pbc and n_qubits > 2:
        zz_gate(dt, n_qubits - 1, 0)      # wrap-around bond

    # --- Layer 2: Next-nearest-neighbor ZZ interactions ---
    # exp(-i * J2 * dt * Z_i Z_{i+2})
    for i in range(n_qubits - 2):
        zz_gate(-J2 * dt, i, i + 2)            # theta = -J2*dt  ←  -(+J2)*dt
    if pbc and n_qubits > 3:
        zz_gate(-J2 * dt, n_qubits - 2, 0)     # wrap-around bonds
        zz_gate(-J2 * dt, n_qubits - 1, 1)

    # --- Layer 3: Transverse field X rotations ---
    # exp(+i * h * dt * X_i) = RX(-2*h*dt)
    for i in range(n_qubits):
        qml.RX(-2 * h * dt, wires=i)


def annni_time_evolution(n_qubits, k, h, t, n_steps, pbc=True):
    dt = t / n_steps
    for _ in range(n_steps):
        annni_trotter_step(n_qubits, k, h, dt, pbc)


In [ ]:
n_qubits = 8
k = 0.8
h = 0.15
inc = 2000
t = 32.0
ts = np.linspace(0, t, inc)
n_steps = 10
dev = qml.device("lightning.gpu", wires=n_qubits)

@qml.qnode(dev)
def circuit(t, k, h):
    # Initialise in |+...+> (eigenstate of X, good for seeing field dynamics)
    #for i in range(n_qubits):
        #qml.Hadamard(wires=i)

    annni_time_evolution(n_qubits, k, h, t, n_steps, pbc=True)

    return qml.state()

vals = np.zeros(inc)

for i in range(inc):
    vals[i] = abs(circuit(ts[i], k, h)[0])**2

rate = -0.125*np.log(vals+1e-14)
plt.plot(ts, rate)


In [ ]:
np.real(exact_ground_state(8, 0.5, 1)["state"])
